# 00. Release Artifact Tour

This notebook reads only release-safe aggregate artifacts. It is useful for checking what numbers support the manuscript without touching private row-level logs.

<!-- PEC_NOTEBOOK_EXPLANATION_V1:release_tour_guide -->

## Reader guide: what this notebook is for

Use this notebook as the **map between the paper and the repository artifacts**. It does not rebuild the private row-level analysis. Instead, it answers three practical questions:

1. Which aggregate tables support each paper claim?
2. Which figures are release-safe and can be regenerated from non-user-level data?
3. Which values should match the manuscript when the repository is in a consistent state?

The most important idea is the PEC separation: **Preference (P)**, **logged Exposure (E)**, and **Click Consumption (C)** are all useful traces, but they are not interchangeable. This notebook checks the released artifacts that summarize those traces.

<!-- PEC_NOTEBOOK_EXPLANATION_V1:release_artifact_terms -->

## Key terms used in this tour

- **Release-safe aggregate table:** a table containing paper-level or cohort-level values, not raw user histories.
- **Traceability:** whether a clicked article can be linked back to a logged recommendation exposure. This is not CTR.
- **Diversity shift:** whether diversity in logged exposure appears again in clicked consumption.
- **Diagnostic model:** a model used to ask whether high-divergence users are structured by observable features. It is not a production recommender or targeting model.

When reading outputs, focus on whether the artifact supports the claim direction and magnitude reported in the paper.

In [ ]:
from pathlib import Path
import os
import sys
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
ROOT

In [ ]:
from pecgap.artifacts import load_json, load_table
analysis = load_json('analysis_summary.json', ROOT)
analysis

## Popularity-aware baseline

The logged recommendation hit rate should be compared not only to a date-matched random candidate pool but also to simple non-personalized popularity baselines. This does not prove causal personalization, but it helps rule out the weakest explanation that recommendations are no better than generic popularity.

In [ ]:
pop = load_table('popularity_baseline_check.csv', ROOT)
pop.head(12)

In [ ]:
topk = pop[pop['k'] == pop['k'].max()][['baseline','hit_pct','actual_minus_baseline_pp']]
topk

## Surface traceability

Surface-specific lift shows whether clicks from different app pathways are equally traceable to logged recommendation exposure.

In [ ]:
surface = load_table('optimization_surface_lift.csv', ROOT)
surface.sort_values('lift', ascending=False)

## Diagnostic model

The diagnostic model is not a targeting system. It checks whether high-divergence users are structured by observable features.

In [ ]:
model = load_table('deep_tuned_model_cv_results.csv', ROOT)
model

In [ ]:
feature_importance = load_table('rq4_feature_importance.csv', ROOT)
feature_importance.head(12)